# Train TrafficCAM loop (Free T4 — the only GPU step)

Fine-tunes `yolov8n` on converted TrafficCAM (+ ITD later, same converter interface) and packages a registry candidate.
The finale stays canonical unless the candidate wins on measurement (gates in Cell 5).

| | |
|---|---|---
| GPU | **Required here.** Runtime → T4. Everything else in this notebook is CPU. |
| Prereq | Run `eval_trafficcam_drive.ipynb` Cells 0–3 first, **or** re-run Cells 0–2 here (idempotent). |
| Upload | `best_f007.pt` (loop-8 weights) to `/content` when asked — same Drive contract as the BMD finale. |
| Hand-back | `trafficcam_candidate_<date>.zip` (fp32 + int8 + metadata.json + val report) → `notebooks/training_output_zips/` |

Resume-safe: re-running prep cells skips existing outputs.

In [ ]:
# Cell 1 — data: download + convert (idempotent; skip if /content/tcam exists)
import os
if not os.path.exists('/content/tcam/data.yaml'):
    # the shell cwd on each line, so relative cd/gdown/unzip silently misplaces files)
    !mkdir -p /content/trafficcam
    !test -f /content/trafficcam/Fully_annotate.zip || gdown https://drive.google.com/uc?id=1h4oUqECDF05vSYMkYgz0aUc_RRzHlh3e -O /content/trafficcam/Fully_annotate.zip
    !test -f /content/trafficcam/splits.zip || gdown https://drive.google.com/uc?id=1GcQ3J56kU-6TwRqRjzvI28-OF6QUdHeR -O /content/trafficcam/splits.zip
    # rescue: an older revision of this cell dropped zips in /content — reuse them
    !test -f /content/Fully_annotate.zip && cp -n /content/Fully_annotate.zip /content/trafficcam/ ; true
    !test -f /content/splits.zip && cp -n /content/splits.zip /content/trafficcam/ ; true
    !unzip -q -o /content/trafficcam/Fully_annotate.zip -d /content/trafficcam
    !unzip -q -o /content/trafficcam/splits.zip -d /content/trafficcam
    import glob
    _n = len(glob.glob('/content/trafficcam/**/frame0.json', recursive=True))
    assert _n >= 78, f'STOP: expected >=78 videos under /content/trafficcam, found {_n}'
    print(f'download ok: {_n} videos')
    !cd /content/SGP-IV && python scripts/prepare_dataset.py /content/trafficcam \
        --out /content/tcam --source trafficcam --option B
    !cd /content/SGP-IV && python scripts/prepare_dataset.py /content/tcam --make-calibration
    !cd /content/SGP-IV && python scripts/prepare_dataset.py /content/tcam --check
    import os as _os
    assert _os.path.exists('/content/tcam/data.yaml'), 'STOP: converter failed — read output above'
else:
    print('tcam exists, skipped')
# ITD later: convert with its own --source into /content/itd, then merge
# train/val/test image+label dirs into /content/tcam (same 6-class ids).


In [ ]:
# Cell 1 — data: download + convert (idempotent; skip if /content/tcam exists)
import os
if not os.path.exists('/content/tcam/data.yaml'):
    !mkdir -p /content/trafficcam && cd /content/trafficcam
    !gdown --id 1h4oUqECDF05vSYMkYgz0aUc_RRzHlh3e -O Fully_annotate.zip
    !unzip -q -o Fully_annotate.zip
    !cd /content/SGP-IV && python scripts/prepare_dataset.py /content/trafficcam \
        --out /content/tcam --source trafficcam --option B
    !cd /content/SGP-IV && python scripts/prepare_dataset.py /content/tcam --make-calibration
    !cd /content/SGP-IV && python scripts/prepare_dataset.py /content/tcam --check
else:
    print('tcam exists, skipped')
# ITD later: convert with its own --source into /content/itd, then merge
# train/val/test image+label dirs into /content/tcam (same 6-class ids).

In [ ]:
# Cell 2 — train: yolov8n from loop-8 weights, low-LR polish (finale recipe)
# Upload best_f007.pt to /content when prompted (Files pane > Upload).
# PREREQ: Cell 1 must have produced /content/tcam/data.yaml (run it first).
import os
assert os.path.exists('/content/best_f007.pt'), 'upload best_f007.pt first'
assert os.path.exists('/content/tcam/data.yaml'), \
    'STOP: run Cell 1 (data download + convert) first — /content/tcam/data.yaml is missing'
from ultralytics import YOLO
model = YOLO('/content/best_f007.pt')
model.train(data='/content/tcam/data.yaml', epochs=10, imgsz=640, batch=32, workers=4,
              lr0=0.002, patience=10, project='/content/runs', name='tcam_loop',
              seed=42, verbose=True)
print('train done')

In [ ]:
# Cell 3 — validate: TrafficCAM val + (optional) BMD-45 no-forgetting anchor
# Anchor: mount Drive with BMD-45 val converted set, set BMD_VAL to its data.yaml.
BMD_VAL = ''  # e.g. '/content/drive/MyDrive/bmd45/data.yaml' — leave empty to skip
import glob
w = sorted(glob.glob('/content/runs/tcam_loop/weights/best.pt'))[-1]
print('weights:', w)
!yolo val model={w} data=/content/tcam/data.yaml split=val 2>&1 | tail -8
if BMD_VAL:
    !yolo val model={w} data={BMD_VAL} split=val 2>&1 | tail -8
    print('GATE: overall mAP50 must stay within -0.02 of 0.8477')

In [ ]:
# Cell 4 — export ONNX opset17 + static int8 (calibration reader over tcam calib frames)
import glob, os
from ultralytics import YOLO
w = sorted(glob.glob('/content/runs/tcam_loop/weights/best.pt'))[-1]
onnx = YOLO(w).export(format='onnx', opset=17, dynamic=False)
print('fp32:', onnx)

import cv2, numpy as np, onnxruntime
from onnxruntime.quantization import CalibrationDataReader, QuantFormat, quantize_static
from onnxruntime.quantization.quantize import QuantType

cals = sorted(glob.glob('/content/tcam/calibration/frames/*.jpg'))[:100]
assert cals, 'no calibration frames'
sess = onnxruntime.InferenceSession(onnx, providers=['CPUExecutionProvider'])
inp = sess.get_inputs()[0].name

class Reader(CalibrationDataReader):
    def __init__(self):
        self.it = iter(cals)
    def get_next(self):
        try:
            p = next(self.it)
        except StopIteration:
            return None
        img = cv2.resize(cv2.imread(p), (640, 640)).astype(np.float32) / 255.0
        return {inp: np.ascontiguousarray(img.transpose(2, 0, 1)[None])}

int8 = onnx.replace('model.onnx', 'model-int8.onnx') if onnx.endswith('.onnx') else onnx + '.int8.onnx'
try:
    quantize_static(onnx, int8, Reader(), quant_format=QuantFormat.QDQ,
                    weight_type=QuantType.QInt8)
    print('static int8:', int8)
except Exception as e:
    print('STATIC QUANT FAILED (dynamic fallback is CPU-slower, do not ship):', e)

In [ ]:
# Cell 5 — package registry candidate + verdict
import datetime, json, os, shutil
stamp = datetime.date.today().isoformat()
reg = f'/content/registry_candidate_{stamp}'
os.makedirs(reg, exist_ok=True)
for f in [onnx, int8]:
    shutil.copy(f, reg)
shutil.copy('/content/tcam/data.yaml', os.path.join(reg, 'data.yaml'))
json.dump({'classes': ['car', 'motorcycle', 'bus', 'truck', 'bicycle', 'auto'],
             'imgsz': 640, 'source': 'trafficcam-tcam_loop', 'base': 'best_f007.pt',
             'date': stamp}, open(os.path.join(reg, 'metadata.json'), 'w'), indent=2)
zipf = f'/content/trafficcam_candidate_{stamp}.zip'
!cd /content && zip -qr {zipf} {reg}
!ls -la {zipf}
from google.colab import files
files.download(zipf)
print('HAND-BACK:', zipf)
print('Promotion gates (applied on return): TrafficCAM mAP50 up AND BMD-Val within -0.02 of 0.8477 AND int8 fps >= 4.5 CPU. Else finale stays canonical.')